In [1]:
import re
import gc
import time

import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler

In [2]:
TRAIN_IN = "/kaggle/input/datasets/zerotoinfinitya/train-preprocessed/train_preprocessed.parquet"
VAL_IN = "/kaggle/input/datasets/zerotoinfinitya/validation-preprocessed/validation_preprocessed.parquet"
TEST_IN = "/kaggle/input/datasets/zerotoinfinitya/test-preprocessed/test_preprocessed.parquet"

TRAIN_OUT = "/kaggle/working/train_structural_features.npz"
VAL_OUT = "/kaggle/working/validation_structural_features.npz"
TEST_OUT = "/kaggle/working/test_structural_features.npz"
SCALER_OUT = "/kaggle/working/scaler_structural.joblib"

In [3]:
RE_FUNC = re.compile(r"\bfunction\b[^{;]*[{;]")
RE_PRAGMA = re.compile(
    r"pragma\s+solidity\s*[^;]*?(\d+)\s*\.\s*(\d+)(?:\s*\.\s*(\d+))?")
RE_MOD = re.compile(r"\bmodifier\s+\w+")
EXT_CALL = re.compile(
    r"\.\s*(call|delegatecall|callcode|staticcall|send|transfer)\s*[({.]")
STATE_WRITE = re.compile(r"\b\w+\s*(\[[^\]]*\]\s*)?(\.\s*\w+\s*)?[-+*/]?=[^=]")
GUARD = re.compile(r"\b(require|assert|revert)\s*\(")

In [4]:
def split_functions(src):
    out = []
    for m in RE_FUNC.finditer(src):
        if src[m.end() - 1] == ";":
            continue                      # declaration only, no body
        depth, j, n = 0, m.end() - 1, len(src)
        while j < n:
            if src[j] == "{":
                depth += 1
            elif src[j] == "}":
                depth -= 1
                if depth == 0:
                    break
            j += 1
        out.append(src[m.start():min(j + 1, n)])
    return out

In [5]:
def pragma_version(src):
    m = RE_PRAGMA.search(src)
    if not m:
        return -1.0
    try:
        a = int(m.group(1))
        b = int(m.group(2))
        c = int(m.group(3)) if m.group(3) else 0
        if a == 0:
            return b + c / 100.0
        return a + b / 100.0
    except Exception:
        return -1.0

In [6]:
STRUCT_NAMES = [
    # plain counts
    "n_func", "n_func_body", "n_public", "n_external", "n_payable",
    "n_view", "n_modifier", "n_require", "n_assert", "n_revert",
    "n_ext_call", "n_lowlevel_call", "n_delegatecall", "n_send",
    "n_transfer", "n_selfdestruct", "n_assembly", "n_loop",
    # ordering, inside one function body
    "call_before_write_funcs", "call_before_write_max",
    "call_before_write_frac", "unguarded_call_funcs",
    "unguarded_call_frac", "call_in_loop_funcs",
    "public_write_no_modifier", "payable_no_withdraw",
    # untrusted / unpredictable sources
    "uses_tx_origin", "tx_origin_in_cond", "uses_timestamp",
    "timestamp_in_cond", "uses_blockhash", "uses_difficulty",
    "rand_from_block",
    # arithmetic safety
    "has_safemath", "pragma_ver", "pragma_ge_08", "pragma_unknown",
    "arith_ops", "arith_unsafe",
    # shape of the file
    "n_state_write", "src_len_k", "trunc_flag", "n_mapping",
    "n_address", "n_uint", "n_inherit", "n_event", "n_emit",
    "n_constructor", "n_fallback", "n_receive", "n_unchecked_block",
]
IX = {n: i for i, n in enumerate(STRUCT_NAMES)}
assert len(STRUCT_NAMES) == 52

In [7]:
def struct_counts(F, r, src, low, funcs):
    F[r, IX["n_func"]] = len(RE_FUNC.findall(src))
    F[r, IX["n_func_body"]] = len(funcs)
    F[r, IX["n_public"]] = src.count("public")
    F[r, IX["n_external"]] = src.count("external")
    F[r, IX["n_payable"]] = src.count("payable")
    F[r, IX["n_view"]] = src.count("view") + src.count("pure")
    F[r, IX["n_modifier"]] = len(RE_MOD.findall(src))
    F[r, IX["n_require"]] = src.count("require(")
    F[r, IX["n_assert"]] = src.count("assert(")
    F[r, IX["n_revert"]] = src.count("revert")
    F[r, IX["n_ext_call"]] = len(EXT_CALL.findall(src))
    F[r, IX["n_lowlevel_call"]] = (low.count(".call(") + low.count(".call{")
                                   + low.count(".call.value"))
    F[r, IX["n_delegatecall"]] = low.count("delegatecall")
    F[r, IX["n_send"]] = low.count(".send(")
    F[r, IX["n_transfer"]] = low.count(".transfer(")
    F[r, IX["n_selfdestruct"]] = (low.count("selfdestruct")
                                  + low.count("suicide("))
    F[r, IX["n_assembly"]] = low.count("assembly")
    F[r, IX["n_loop"]] = (src.count("for (") + src.count("for(")
                          + src.count("while (") + src.count("while("))
    F[r, IX["n_state_write"]] = len(STATE_WRITE.findall(src))
    F[r, IX["n_mapping"]] = src.count("mapping")
    F[r, IX["n_address"]] = src.count("address")
    F[r, IX["n_uint"]] = src.count("uint")
    F[r, IX["n_inherit"]] = src.count(" is ")
    F[r, IX["n_event"]] = src.count("event ")
    F[r, IX["n_emit"]] = src.count("emit ")
    F[r, IX["n_constructor"]] = src.count("constructor")
    F[r, IX["n_fallback"]] = src.count("fallback")
    F[r, IX["n_receive"]] = src.count("receive(")
    F[r, IX["n_unchecked_block"]] = src.count("unchecked")

In [8]:
def struct_order(F, r, funcs):
    cbw, unguarded, in_loop, pub_no_mod, cbw_max = 0, 0, 0, 0, 0
    for fb in funcs:
        calls = [m.start() for m in EXT_CALL.finditer(fb)]
        writes = [m.start() for m in STATE_WRITE.finditer(fb)]
        head_ = fb[:fb.find("{") + 1] if "{" in fb else fb
        if (("public" in head_ or "external" in head_)
                and "only" not in head_.lower()
                and "view" not in head_ and "pure" not in head_
                and writes):
            pub_no_mod += 1
        if not calls:
            continue
        # a state write AFTER an external call is the reentrancy shape
        first_call = min(calls)
        after = sum(1 for w in writes if w > first_call)
        if after > 0:
            cbw += 1
            cbw_max = max(cbw_max, after)
        if not GUARD.search(fb):
            unguarded += 1
        if "for" in fb or "while" in fb:
            in_loop += 1
    nf = max(1, len(funcs))
    F[r, IX["call_before_write_funcs"]] = cbw
    F[r, IX["call_before_write_max"]] = cbw_max
    F[r, IX["call_before_write_frac"]] = cbw / nf
    F[r, IX["unguarded_call_funcs"]] = unguarded
    F[r, IX["unguarded_call_frac"]] = unguarded / nf
    F[r, IX["call_in_loop_funcs"]] = in_loop
    F[r, IX["public_write_no_modifier"]] = pub_no_mod

In [9]:
def struct_taint(F, r, src, low):
    pays_out = (low.count(".transfer(") + low.count(".send(")
                + low.count(".call") + low.count("selfdestruct"))
    F[r, IX["payable_no_withdraw"]] = 1.0 if ("payable" in low
                                              and pays_out == 0) else 0.0
    F[r, IX["uses_tx_origin"]] = low.count("tx.origin")
    F[r, IX["uses_timestamp"]] = (low.count("block.timestamp")
                                  + len(re.findall(r"\bnow\b", low)))
    F[r, IX["uses_blockhash"]] = low.count("blockhash")
    F[r, IX["uses_difficulty"]] = (low.count("block.difficulty")
                                   + low.count("prevrandao"))
    cond = " ".join(re.findall(r"(?:require|assert|if)\s*\(([^;{]*)", low))
    F[r, IX["tx_origin_in_cond"]] = cond.count("tx.origin")
    F[r, IX["timestamp_in_cond"]] = (cond.count("block.timestamp")
                                     + len(re.findall(r"\bnow\b", cond)))
    kec = re.findall(r"keccak256\s*\(([^;)]*)", low)
    F[r, IX["rand_from_block"]] = sum(
        1 for k in kec if ("timestamp" in k or "blockhash" in k
                           or "difficulty" in k or "now" in k))

In [10]:
def struct_arith(F, r, src, low):
    sm = 1.0 if "safemath" in low else 0.0
    ver = pragma_version(src)
    arith = (src.count(" + ") + src.count(" - ") + src.count(" * ")
             + src.count("+=") + src.count("-=") + src.count("*="))
    F[r, IX["has_safemath"]] = sm
    F[r, IX["pragma_ver"]] = ver
    F[r, IX["pragma_ge_08"]] = 1.0 if ver >= 8.0 else 0.0
    F[r, IX["pragma_unknown"]] = 1.0 if ver < 0 else 0.0
    F[r, IX["arith_ops"]] = arith
    # Solidity >= 0.8 reverts on overflow by itself
    unsafe_ver = ver < 0 or ver < 8.0
    F[r, IX["arith_unsafe"]] = arith if (unsafe_ver and sm == 0) else 0.0

In [11]:
def struct_features(texts, trunc):
    F = np.zeros((len(texts), 52), dtype=np.float32)
    start = time.time()
    for r in range(len(texts)):
        src = texts[r]
        if not src:
            continue
        low = src.lower()
        funcs = split_functions(src)
        struct_counts(F, r, src, low, funcs)
        struct_order(F, r, funcs)
        struct_taint(F, r, src, low)
        struct_arith(F, r, src, low)
        F[r, IX["src_len_k"]] = len(src) / 1000.0
        F[r, IX["trunc_flag"]] = 1.0 if trunc[r] else 0.0
        if r and r % 20000 == 0:
            print("   %d / %d   %d s" % (r, len(texts), time.time() - start))
    return F

In [12]:
def read_split(path):
    df = pd.read_parquet(path, columns=["source_capped", "trunc_flag"])
    texts = ["" if t is None else t for t in df["source_capped"]]
    trunc = df["trunc_flag"].values.astype(bool)
    return texts, trunc

In [13]:

texts, trunc = read_split(TRAIN_IN)
train_raw = struct_features(texts, trunc)
example_source = texts[0]             
del texts, trunc
gc.collect()

   20000 / 79641   200 s
   40000 / 79641   414 s
   60000 / 79641   649 s


30

In [15]:
texts, trunc = read_split(VAL_IN)
val_raw = struct_features(texts, trunc)
del texts, trunc
gc.collect()

0

In [16]:
texts, trunc = read_split(TEST_IN)
test_raw = struct_features(texts, trunc)
del texts, trunc
gc.collect()

0

In [17]:
scaler = StandardScaler()
scaler.fit(train_raw)

train_struct = scaler.transform(train_raw).astype(np.float32)
val_struct = scaler.transform(val_raw).astype(np.float32)
test_struct = scaler.transform(test_raw).astype(np.float32)

constant = [STRUCT_NAMES[j] for j in range(52) if train_raw[:, j].std() < 1e-9]

In [18]:
names = np.asarray(STRUCT_NAMES, dtype=np.str_)
np.savez_compressed(TRAIN_OUT, struct=train_struct, names=names)
np.savez_compressed(VAL_OUT, struct=val_struct, names=names)
np.savez_compressed(TEST_OUT, struct=test_struct, names=names)
joblib.dump({"scaler": scaler, "names": STRUCT_NAMES}, SCALER_OUT)
print("Saved:")
print("  ", TRAIN_OUT)
print("  ", VAL_OUT)
print("  ", TEST_OUT)
print("  ", SCALER_OUT)

Saved:
   /kaggle/working/train_structural_features.npz
   /kaggle/working/validation_structural_features.npz
   /kaggle/working/test_structural_features.npz
   /kaggle/working/scaler_structural.joblib


In [21]:
print("   train     :", train_struct.shape)
print("   validation:", val_struct.shape)
print("   test      :", test_struct.shape)

print()
print("total contracts:", len(train_raw) + len(val_raw) + len(test_raw))


   train     : (79641, 52)
   validation: (10861, 52)
   test      : (15972, 52)

total contracts: 106474
